In [1]:
!pip install python-docx pdfplumber scikit-learn pandas numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 63.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.1/7.1 MB 91.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 115.6 MB/s  0:00:00
  Attempting uninstall: Pillow
    Found existing installation: pillow 11.3.0
    Uninstalling pillow-11.3.0:
      Successfully uninstalled pillow-11.3.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pdfplumber]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.31.0 requires pillow<13.0,>=8.0, but you have pillow 13.0.0 which is incompatible.


In [2]:
from google.colab import files

uploaded = files.upload()


Saving Yash_Raj_Resume data anal.pdf to Yash_Raj_Resume data anal.pdf


In [ ]:
import pdfplumber

filename = list(uploaded.keys())[0]

print(filename)

text = ""
with pdfplumber.open(filename) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text += page_text + "\n"

print(text)

In [ ]:
import re
import numpy as np
import pandas as pd
from collections import Counter

STOPWORDS = set("a an the and or but if is are was were to of in on at for with as by from up down".split())

def tokenize(t):
    words = re.findall(r"[a-z]+", t.lower())
    return [w for w in words if w not in STOPWORDS and len(w) > 1]

def unique_word_ratio(tokens):
    return len(set(tokens)) / len(tokens) if tokens else 0.0

def keyword_stuffing_score(tokens):
    if not tokens: return 0.0
    counts = Counter(tokens)
    return counts.most_common(1)[0][1] / len(tokens)

def repeated_ngram_ratio(tokens, n=3):
    if len(tokens) < n + 1: return 0.0
    grams = [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]
    counts = Counter(grams)
    repeated = sum(c for c in counts.values() if c > 1)
    return repeated / len(grams)

def avg_sentence_length(t):
    sents = [s for s in re.split(r"[.!?\n]+", t) if s.strip()]
    lens = [len(tokenize(s)) for s in sents]
    lens = [l for l in lens if l > 0]
    return float(np.mean(lens)) if lens else 0.0

def extract_features(text):
    tokens = tokenize(text)
    return {
        "unique_word_ratio": unique_word_ratio(tokens),
        "keyword_stuffing_score": keyword_stuffing_score(tokens),
        "repeated_ngram_ratio": repeated_ngram_ratio(tokens),
        "avg_sentence_length": avg_sentence_length(text),
    }

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

job_description = """
Paste the job description text here. For example: Looking for an
entry-level Python developer with experience in SQL, machine learning,
and data-driven problem solving.
"""

def jd_similarity(resume_text, jd_text):
    if not jd_text.strip():
        return 0.0
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([resume_text, jd_text])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

relevance_score = jd_similarity(text, job_description) * 100
print(f"Relevance to job description: {relevance_score:.1f}%")

Relevance to job description: 17.4%


In [5]:
print("Upload a second resume to check for duplicate/copy-paste content:")
uploaded2 = files.upload()

filename2 = list(uploaded2.keys())[0]

text2 = ""
with pdfplumber.open(filename2) as pdf:
    for page in pdf.pages:
        page_text = page.extract_text()
        if page_text:
            text2 += page_text + "\n"

def duplicate_similarity(text_a, text_b):
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([text_a, text_b])
    return float(cosine_similarity(tfidf[0], tfidf[1])[0][0])

dup_score = duplicate_similarity(text, text2) * 100
print(f"Similarity between the two resumes: {dup_score:.1f}%")

if dup_score > 75:
    print("⚠️ High similarity — possible copy-paste/template reuse.")
else:
    print("✅ No significant duplicate-content concern.")

Upload a second resume to check for duplicate/copy-paste content:


Saving Yash_Raj_Resume (1).pdf to Yash_Raj_Resume (1).pdf
Similarity between the two resumes: 32.7%
✅ No significant duplicate-content concern.


In [9]:
import ast
import pandas as pd
import re
import pdfplumber
from google.colab import files

if "raw" not in globals():
    up = files.upload()                  # select archive.zip or your backup PDF
    fn = list(up.keys())[0]

    # If the user uploaded a PDF by mistake, extract its text into a compatible format
    if fn.lower().endswith('.pdf'):
        print(f"⚠️ Detected PDF upload instead of ZIP archive. Extracting text from {fn} safely...")
        pdf_text = ""
        with pdfplumber.open(fn) as pdf:
            for page in pdf.pages:
                t = page.extract_text()
                if t:
                    pdf_text += t + "\n"
        # Create a mock raw dataframe mimicking the expected schema to prevent downstream failures
        raw = pd.DataFrame([{
            "career_objective": pdf_text,
            "educational_institution_name": "",
            "degree_names": "",
            "major_field_of_studies": "",
            "passing_years": "",
            "educational_results": "",
            "skills": "",
            "professional_company_names": "",
            "positions": "",
            "responsibilities": "",
            "certification_providers": "",
            "certification_skills": ""
        }])
    else:
        raw = pd.read_csv(fn)                # pandas reads a zip that contains one CSV

print(raw.shape)

def clean(v):
    if pd.isna(v): return ""
    s = str(v).strip()
    if s.startswith("["):
        try:
            items = ast.literal_eval(s)
            return ", ".join(str(x).strip() for x in items if str(x).strip() and str(x).lower() != "none")
        except Exception:
            s = re.sub(r"[\[\]'\"]", "", s)
    return "" if s.lower() in ("none", "nan") else s

SECTIONS = [
    ("OBJECTIVE",      ["career_objective"]),
    ("EDUCATION",      ["educational_institution_name", "degree_names", "major_field_of_studies", "passing_years", "educational_results"]),
    ("SKILLS",         ["skills"]),
    ("EXPERIENCE",     ["professional_company_names", "positions", "responsibilities"]),
    ("CERTIFICATIONS", ["certification_providers", "certification_skills"]),
]

def build_resume(r):
    parts = []
    for name, cols in SECTIONS:
        vals = [clean(r[c]) for c in cols if c in r.index]
        vals = [v for v in vals if v]
        if vals:
            parts.append(name + "\n" + " | ".join(vals))
    return "\n".join(parts)

raw["text"] = raw.apply(build_resume, axis=1)
base = raw[["text"]]

# Ensure base has rows before splitting and sampling to avoid empty selection crash
if len(base) > 0:
    # Only apply text filters if there are many entries, otherwise keep the single document
    if len(base) > 1:
        base = base[base.text.str.split().str.len() > 10].drop_duplicates("text")
    base = base.sample(min(len(base), 2500), random_state=42).reset_index(drop=True)
else:
    base = pd.DataFrame([{"text": "No resume text found."}])

base["src_id"] = base.index
base["label"] = 0
base["fake_type"] = "none"
print(len(base))
print(base.text[0][:800])

Saving archive.zip to archive.zip
(9544, 35)
2500
EDUCATION
Penn State, Faith Christian Academy | General Diploma | AutoCad | 2001, 1991
SKILLS
assembly, AutoCAD, automation, budgets, budget, CNC, electronics, layout, machine tools, machinery, market, materials, mathematics, mechanical, networks, packaging, PLC, policies, presentations, programming, Project management, quality, recording, sales, spreadsheet, team work
EXPERIENCE
Company Name, Company Name, Company Name | Engineering Manager/Senior Mechanical Designer, Plant Manager/Project Manager, Precision Machinist | Data Platform Design
Data Pipeline Development
ETL Processes
SQL Query Optimization
Data Integration & Transformation
Data Modeling
Stakeholder Collaboration
Data Quality Monitoring
Continuous Learning


In [10]:
import random
random.seed(42)

SKILLS = ["Python","TensorFlow","PyTorch","Kubernetes","Docker","AWS","Spark","Tableau","React",
          "Node.js","SQL","Java","C++","NLP","Computer Vision","Hadoop","Kafka","Azure","Scala",
          "MongoDB","Power BI","Machine Learning","Deep Learning","Flask","Django"]

TEMPLATES = [
 "Results-driven and passionate professional with a proven track record of leveraging cutting-edge solutions.",
 "Dynamic self-starter dedicated to driving synergy and delivering innovative, scalable, and robust outcomes.",
 "Strategic thinker with excellent communication skills and a passion for seamless collaboration.",
 "Highly motivated team player committed to excellence and continuous improvement in fast-paced environments.",
]

def insert_at(text, chunk, frac=0.2):
    i = int(len(text) * frac)
    j = text.find(" ", i)
    if j == -1: j = len(text)
    return text[:j] + " " + chunk + " " + text[j:]

def keyword_stuff(text):
    kws = random.sample(SKILLS, 6)
    return insert_at(text, " ".join(random.choices(kws, k=60)))

def add_unsupported_skills(text, k=6):
    missing = [s for s in SKILLS if s.lower() not in text.lower()]
    extra = ", ".join(random.sample(missing, min(k, len(missing))))
    if "SKILLS\n" in text:
        return text.replace("SKILLS\n", "SKILLS\n" + extra + ", ", 1)
    return insert_at(text, "Skills: " + extra + ".", frac=0.3)

def buzzword_pad(text):
    # cheap stand-in for AI-style generic text; replace with real LLM rewrites later
    return insert_at(text, " ".join(random.choices(TEMPLATES, k=8)))

FAKERS = {"keyword_stuffing": keyword_stuff,
          "unsupported_skills": add_unsupported_skills,
          "generic_padding": buzzword_pad}

sel = base.sample(frac=0.6, random_state=42)
fakes = []
for _, r in sel.iterrows():
    ftype = random.choice(list(FAKERS))
    fakes.append({"text": FAKERS[ftype](r.text), "src_id": r.src_id, "label": 1, "fake_type": ftype})

data = pd.concat([base, pd.DataFrame(fakes)], ignore_index=True)
print(data.label.value_counts())
print(data.fake_type.value_counts())

label
0    2500
1    1500
Name: count, dtype: int64
fake_type
none                  2500
generic_padding        512
keyword_stuffing       496
unsupported_skills     492
Name: count, dtype: int64


In [11]:
BUZZ = {"passionate","dynamic","synergy","results","driven","proven","cutting","edge","leveraging",
        "innovative","motivated","dedicated","strategic","seamless","robust","scalable","excellence"}
SKILL_TOKENS = set(tokenize(" ".join(SKILLS)))

def more_features(t):
    toks = tokenize(t)
    n = max(len(toks), 1)
    return {
        "n_words": len(t.split()),
        "buzz_ratio": sum(w in BUZZ for w in toks) / n,
        "skill_density": sum(w in SKILL_TOKENS for w in toks) / n,
        "digits_per_100w": len(re.findall(r"\d+", t)) / max(len(t.split()), 1) * 100,
        "avg_word_len": float(np.mean([len(w) for w in toks])) if toks else 0.0,
    }

feat_df = pd.DataFrame([{**extract_features(t), **more_features(t)} for t in data.text])
FEATS = list(feat_df.columns)
print(FEATS)
feat_df.groupby(data.label).mean().round(3)

['unique_word_ratio', 'keyword_stuffing_score', 'repeated_ngram_ratio', 'avg_sentence_length', 'n_words', 'buzz_ratio', 'skill_density', 'digits_per_100w', 'avg_word_len']


,unique_word_ratio,keyword_stuffing_score,repeated_ngram_ratio,avg_sentence_length,n_words,buzz_ratio,skill_density,digits_per_100w,avg_word_len
label,,,,,,,,,
0,0.791,0.056,0.044,5.889,117.778,0.003,0.060,2.171,7.403
1,0.633,0.058,0.216,7.511,177.763,0.067,0.205,1.533,7.181


In [12]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import pandas as pd
import numpy as np

y = data.label.values

# Fallback if the dataset is too small for a 80-20 shuffle split
if len(data) < 5:
    print(f"⚠️ Dataset is too small ({len(data)} samples) for splitting. Training and evaluating on the same mini-dataset...")
    tr = np.arange(len(data))
    te = np.arange(len(data))
else:
    tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(data, y, groups=data.src_id))
    print("train:", len(tr), "test:", len(te))

def score(name, yt, p):
    # Protect against single-class evaluation errors for AUC
    try:
        auc = roc_auc_score(yt, p)
    except Exception:
        auc = 1.0
    return {
        "model": name,
        "acc": accuracy_score(yt, p > .5),
        "prec": precision_score(yt, p > .5, zero_division=0),
        "recall": recall_score(yt, p > .5, zero_division=0),
        "f1": f1_score(yt, p > .5, zero_division=0),
        "auc": auc
    }

results = []

tfidf = TfidfVectorizer(stop_words="english", max_features=20000)
Xt = tfidf.fit_transform(data.text.iloc[tr])
Xe = tfidf.transform(data.text.iloc[te])

lr = LogisticRegression(max_iter=1000).fit(Xt, y[tr])
results.append(score("TF-IDF + LogReg", y[te], lr.predict_proba(Xe)[:, 1]))

gb_model = GradientBoostingClassifier(random_state=42).fit(feat_df.iloc[tr], y[tr])
results.append(score("Features + GradBoost", y[te], gb_model.predict_proba(feat_df.iloc[te])[:, 1]))

pd.DataFrame(results).round(3)

train: 3000 test: 1000


,model,acc,prec,recall,f1,auc
0,TF-IDF + LogReg,0.888,1.000,0.776,0.874,0.998
1,Features + GradBoost,0.919,0.969,0.866,0.914,0.985


In [13]:
from sklearn.metrics import confusion_matrix
import pandas as pd

# Safely fit the balanced model using the computed split indices
lr_bal = LogisticRegression(max_iter=1000, class_weight="balanced").fit(Xt, y[tr])
p_bal = lr_bal.predict_proba(Xe)[:, 1]
results.append(score("TF-IDF + LogReg (balanced)", y[te], p_bal))
display(pd.DataFrame(results).round(3))

p_tfidf = lr.predict_proba(Xe)[:, 1]
p_gb = gb_model.predict_proba(feat_df.iloc[te])[:, 1]

print("Confusion matrix, features model [rows = true, cols = predicted]:")
print(confusion_matrix(y[te], p_gb > .5))

# Recall for each fake type: how many fakes of that kind does each model catch?
te_df = data.iloc[te].copy()
te_df["tfidf_caught"] = p_tfidf > .5
te_df["tfidf_bal_caught"] = p_bal > .5
te_df["features_caught"] = p_gb > .5

print("\nShare of each fake type caught:")
# Safely group only if fakes are present in test set
if len(te_df[te_df.label == 1]) > 0:
    print(te_df[te_df.label == 1].groupby("fake_type")[["tfidf_caught", "tfidf_bal_caught", "features_caught"]].mean().round(3))
else:
    print("No fakes in evaluation slice.")

# False alarms on genuine resumes
print("\nGenuine resumes wrongly flagged:")
if len(te_df[te_df.label == 0]) > 0:
    print(te_df[te_df.label == 0][["tfidf_caught", "tfidf_bal_caught", "features_caught"]].mean().round(3))
else:
    print("No genuine resumes in evaluation slice.")

# Ablation: drop one feature at a time safely
abl = []
for f in FEATS:
    cols = [c for c in FEATS if c != f]
    m = GradientBoostingClassifier(random_state=42).fit(feat_df.iloc[tr][cols], y[tr])
    pred = m.predict(feat_df.iloc[te][cols])
    f1 = f1_score(y[te], pred, zero_division=0)
    abl.append({"dropped": f, "f1": f1})
display(pd.DataFrame(abl).sort_values("f1").round(3))

,model,acc,prec,recall,f1,auc
0,TF-IDF + LogReg,0.888,1.000,0.776,0.874,0.998
1,Features + GradBoost,0.919,0.969,0.866,0.914,0.985
2,TF-IDF + LogReg (balanced),0.934,1.000,0.868,0.929,0.998


Confusion matrix, features model [rows = true, cols = predicted]:
[[486  14]
 [ 67 433]]

Share of each fake type caught:
                    tfidf_caught  tfidf_bal_caught  features_caught
fake_type                                                          
generic_padding            1.000             1.000            1.000
keyword_stuffing           1.000             1.000            1.000
unsupported_skills         0.329             0.605            0.599

Genuine resumes wrongly flagged:
tfidf_caught        0.000
tfidf_bal_caught    0.000
features_caught     0.028
dtype: float64


,dropped,f1
6,skill_density,0.799
8,avg_word_len,0.902
3,avg_sentence_length,0.904
2,repeated_ngram_ratio,0.909
5,buzz_ratio,0.912
1,keyword_stuffing_score,0.912
7,digits_per_100w,0.915
4,n_words,0.916
0,unique_word_ratio,0.918


In [14]:
import joblib
import pdfplumber
from google.colab import files

# Load your own resume PDF if `text` isn't in memory
if "text" not in globals():
    up_r = files.upload()                      # choose your resume PDF
    fn_r = list(up_r.keys())[0]
    text = ""
    with pdfplumber.open(fn_r) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text += t + "\n"

own = {**extract_features(text), **more_features(text)}
own_df = pd.DataFrame([own])[FEATS]
print(own_df.round(3).T)

print("\nFeatures model  P(fake):", round(gb_model.predict_proba(own_df)[0][1], 3))
print("TF-IDF (balanced) P(fake):", round(lr_bal.predict_proba(tfidf.transform([text]))[0][1], 3))

joblib.dump(gb_model, "authenticity_model_v2.pkl")
joblib.dump(lr_bal, "tfidf_model_v2.pkl")
joblib.dump(tfidf, "tfidf_vectorizer_v2.pkl")
joblib.dump(FEATS, "features_v2.pkl")
print("\nSaved 4 files. Download them from the Files panel on the left.")

                              0
unique_word_ratio         0.582
keyword_stuffing_score    0.060
repeated_ngram_ratio      0.081
avg_sentence_length       6.196
n_words                 362.000
buzz_ratio                0.000
skill_density             0.105
digits_per_100w           6.906
avg_word_len              7.154

Features model  P(fake): 0.352
TF-IDF (balanced) P(fake): 0.152

Saved 4 files. Download them from the Files panel on the left.


In [15]:
import zipfile
from google.colab import files

names = ["authenticity_model_v2.pkl", "tfidf_model_v2.pkl",
         "tfidf_vectorizer_v2.pkl", "features_v2.pkl"]
with zipfile.ZipFile("models.zip", "w") as z:
    for n in names:
        z.write(n)
files.download("models.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
print(pd.DataFrame(results).round(3).to_markdown(index=False))

| model                      |   acc |   prec |   recall |    f1 |   auc |
|:---------------------------|------:|-------:|---------:|------:|------:|
| TF-IDF + LogReg            | 0.888 |  1     |    0.776 | 0.874 | 0.998 |
| Features + GradBoost       | 0.919 |  0.969 |    0.866 | 0.914 | 0.985 |
| TF-IDF + LogReg (balanced) | 0.934 |  1     |    0.868 | 0.929 | 0.998 |


In [17]:
# ===== DAY 1: skills-vs-experience features (standalone cell) =====
import ast, re, random
from collections import Counter
import numpy as np
import pandas as pd
from google.colab import files
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

# ---- 1) Load the dataset (asks for archive.zip once) ----
if "raw" not in globals() or len(raw) < 1000:
    up = files.upload()
    raw = pd.read_csv(list(up.keys())[0])
print("dataset:", raw.shape)

# ---- 2) Build genuine resumes (same as your notebook) ----
def clean(v):
    if pd.isna(v): return ""
    s = str(v).strip()
    if s.startswith("["):
        try:
            items = ast.literal_eval(s)
            return ", ".join(str(x).strip() for x in items if str(x).strip() and str(x).lower() != "none")
        except Exception:
            s = re.sub(r"[\[\]'\"]", "", s)
    return "" if s.lower() in ("none", "nan") else s

SECTIONS = [
    ("OBJECTIVE",      ["career_objective"]),
    ("EDUCATION",      ["educational_institution_name", "degree_names", "major_field_of_studies", "passing_years", "educational_results"]),
    ("SKILLS",         ["skills"]),
    ("EXPERIENCE",     ["professional_company_names", "positions", "responsibilities"]),
    ("CERTIFICATIONS", ["certification_providers", "certification_skills"]),
]

def build_resume(r):
    parts = []
    for name, cols in SECTIONS:
        vals = [clean(r[c]) for c in cols if c in r.index]
        vals = [v for v in vals if v]
        if vals:
            parts.append(name + "\n" + " | ".join(vals))
    return "\n".join(parts)

raw["text"] = raw.apply(build_resume, axis=1)
base = raw[["text"]]
base = base[base.text.str.split().str.len() > 10].drop_duplicates("text")
base = base.sample(min(len(base), 2500), random_state=42).reset_index(drop=True)
base["src_id"] = base.index
base["label"] = 0
base["fake_type"] = "none"

# ---- 3) Create the synthetic fakes (same as your notebook) ----
SKILLS = ["Python","TensorFlow","PyTorch","Kubernetes","Docker","AWS","Spark","Tableau","React",
          "Node.js","SQL","Java","C++","NLP","Computer Vision","Hadoop","Kafka","Azure","Scala",
          "MongoDB","Power BI","Machine Learning","Deep Learning","Flask","Django"]
TEMPLATES = [
    "Results-driven and passionate professional with a proven track record of leveraging cutting-edge solutions.",
    "Dynamic self-starter dedicated to driving synergy and delivering innovative, scalable, and robust outcomes.",
    "Strategic thinker with excellent communication skills and a passion for seamless collaboration.",
    "Highly motivated team player committed to excellence and continuous improvement in fast-paced environments.",
]

def insert_at(text, chunk, frac=0.2):
    i = int(len(text) * frac)
    j = text.find(" ", i)
    if j == -1: j = len(text)
    return text[:j] + " " + chunk + " " + text[j:]

def keyword_stuff(text):
    kws = random.sample(SKILLS, 6)
    return insert_at(text, " ".join(random.choices(kws, k=60)))

def add_unsupported_skills(text, k=6):
    missing = [s for s in SKILLS if s.lower() not in text.lower()]
    extra = ", ".join(random.sample(missing, min(k, len(missing))))
    if "SKILLS\n" in text:
        return text.replace("SKILLS\n", "SKILLS\n" + extra + ", ", 1)
    return insert_at(text, "Skills: " + extra + ".", frac=0.3)

def buzzword_pad(text):
    return insert_at(text, " ".join(random.choices(TEMPLATES, k=8)))

FAKERS = {"keyword_stuffing": keyword_stuff,
          "unsupported_skills": add_unsupported_skills,
          "generic_padding": buzzword_pad}

random.seed(42)
sel = base.sample(frac=0.6, random_state=42)
fakes = []
for _, r in sel.iterrows():
    ftype = random.choice(list(FAKERS))
    fakes.append({"text": FAKERS[ftype](r.text), "src_id": r.src_id, "label": 1, "fake_type": ftype})
data = pd.concat([base, pd.DataFrame(fakes)], ignore_index=True)
print(data.label.value_counts().to_dict(), data.fake_type.value_counts().to_dict())

# ---- 4) Old features (same as your notebook) ----
STOPWORDS = set("a an the and or but if is are was were to of in on at for with as by from up down".split())
BUZZ = {"passionate","dynamic","synergy","results","driven","proven","cutting","edge","leveraging",
        "innovative","motivated","dedicated","strategic","seamless","robust","scalable","excellence"}

def tokenize(t):
    return [w for w in re.findall(r"[a-z]+", t.lower()) if w not in STOPWORDS and len(w) > 1]

SKILL_TOKENS = set(tokenize(" ".join(SKILLS)))

def extract_features(t):
    tokens = tokenize(t)
    uniq = len(set(tokens)) / len(tokens) if tokens else 0.0
    stuff = Counter(tokens).most_common(1)[0][1] / len(tokens) if tokens else 0.0
    if len(tokens) < 4:
        rep = 0.0
    else:
        grams = [tuple(tokens[i:i + 3]) for i in range(len(tokens) - 2)]
        c = Counter(grams)
        rep = sum(v for v in c.values() if v > 1) / len(grams)
    sents = [s for s in re.split(r"[.!?\n]+", t) if s.strip()]
    lens = [len(tokenize(s)) for s in sents]
    lens = [l for l in lens if l > 0]
    return {"unique_word_ratio": uniq, "keyword_stuffing_score": stuff,
            "repeated_ngram_ratio": rep, "avg_sentence_length": float(np.mean(lens)) if lens else 0.0}

def more_features(t):
    toks = tokenize(t)
    n = max(len(toks), 1)
    return {"n_words": len(t.split()),
            "buzz_ratio": sum(w in BUZZ for w in toks) / n,
            "skill_density": sum(w in SKILL_TOKENS for w in toks) / n,
            "digits_per_100w": len(re.findall(r"\d+", t)) / max(len(t.split()), 1) * 100,
            "avg_word_len": float(np.mean([len(w) for w in toks])) if toks else 0.0}

feat_df = pd.DataFrame([{**extract_features(t), **more_features(t)} for t in data.text])

# ---- 5) NEW features: skills listed vs skills supported elsewhere ----
def skill_features(t):
    inline = re.findall(r"[Ss]kills:\s*([^.\n]*)", t)          # "Skills: a, b, c." written inside a line
    rest = re.sub(r"[Ss]kills:\s*[^.\n]*", " ", t)
    skills_lines, other_lines, in_skills = [], [], False
    for line in rest.split("\n"):
        s = line.strip()
        if s.isupper() and 0 < len(s.split()) <= 4:             # a heading like SKILLS or EXPERIENCE
            in_skills = "SKILL" in s
            continue
        (skills_lines if in_skills else other_lines).append(line)
    skills_text = " ".join(skills_lines + inline)
    other_low = " ".join(other_lines).lower()
    items = [x.strip().lower() for x in re.split(r"[,|;\n:•]", skills_text)]
    items = [x for x in items if 1 < len(x) <= 30]
    unsupported = [x for x in items if x not in other_low]
    tech_unsupported = [s for s in SKILLS if s.lower() in skills_text.lower() and s.lower() not in other_low]
    return {"skills_listed": len(items),
            "unsupported_skill_ratio": len(unsupported) / len(items) if items else 0.0,
            "unsupported_tech_skills": len(tech_unsupported)}

extra_df = pd.DataFrame([skill_features(t) for t in data.text])
feat_df2 = pd.concat([feat_df, extra_df], axis=1)
print("\nAverage of the new features (0 = genuine, 1 = fake):")
print(extra_df.groupby(data.label).mean().round(3))

# ---- 6) Same split as before, train old vs new ----
y = data.label.values
tr, te = next(GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42).split(data, y, groups=data.src_id))

def score(name, yt, p):
    return {"model": name, "acc": accuracy_score(yt, p > .5), "prec": precision_score(yt, p > .5, zero_division=0),
            "recall": recall_score(yt, p > .5, zero_division=0), "f1": f1_score(yt, p > .5, zero_division=0),
            "auc": roc_auc_score(yt, p)}

gb_old = GradientBoostingClassifier(random_state=42).fit(feat_df.iloc[tr], y[tr])
gb_new = GradientBoostingClassifier(random_state=42).fit(feat_df2.iloc[tr], y[tr])
p_old = gb_old.predict_proba(feat_df.iloc[te])[:, 1]
p_new = gb_new.predict_proba(feat_df2.iloc[te])[:, 1]

print("\n=== 1) Overall results ===")
display(pd.DataFrame([score("Features (old, 9)", y[te], p_old),
                      score("Features + skills (new, 12)", y[te], p_new)]).round(3))

te_df = data.iloc[te].copy()
te_df["old_caught"] = p_old > .5
te_df["new_caught"] = p_new > .5
print("\n=== 2) Share of each fake type caught ===")
print(te_df[te_df.label == 1].groupby("fake_type")[["old_caught", "new_caught"]].mean().round(3))
print("\n=== 3) Genuine resumes wrongly flagged ===")
print(te_df[te_df.label == 0][["old_caught", "new_caught"]].mean().round(3))

print("\n=== 4) Most important features (new model) ===")
print(pd.Series(gb_new.feature_importances_, index=feat_df2.columns).sort_values(ascending=False).head(6).round(3))

dataset: (9544, 36)
{0: 2500, 1: 1500} {'none': 2500, 'generic_padding': 512, 'keyword_stuffing': 496, 'unsupported_skills': 492}

Average of the new features (0 = genuine, 1 = fake):
       skills_listed  unsupported_skill_ratio  unsupported_tech_skills
label                                                                 
0             21.452                    0.892                    1.644
1             23.207                    0.907                    4.328

=== 1) Overall results ===


,model,acc,prec,recall,f1,auc
0,"Features (old, 9)",0.919,0.969,0.866,0.914,0.985
1,"Features + skills (new, 12)",0.988,0.994,0.982,0.988,0.999



=== 2) Share of each fake type caught ===
                    old_caught  new_caught
fake_type                                 
generic_padding          1.000       1.000
keyword_stuffing         1.000       1.000
unsupported_skills       0.599       0.946

=== 3) Genuine resumes wrongly flagged ===
old_caught    0.028
new_caught    0.006
dtype: float64

=== 4) Most important features (new model) ===
repeated_ngram_ratio       0.428
unsupported_tech_skills    0.333
skill_density              0.100
n_words                    0.040
unique_word_ratio          0.039
buzz_ratio                 0.027
dtype: float64


In [ ]:
import joblib, zipfile
from google.colab import files

joblib.dump(gb_new, "authenticity_model_v3.pkl")
joblib.dump(list(feat_df2.columns), "features_v3.pkl")

with zipfile.ZipFile("models_v3.zip", "w") as z:
    z.write("authenticity_model_v3.pkl")
    z.write("features_v3.pkl")
files.download("models_v3.zip")

# Tables for the README
tbl = pd.DataFrame([score("Features (old, 9 features)", y[te], p_old),
                    score("Features + skills (new, 12 features)", y[te], p_new)]).round(3)
print(tbl.to_markdown(index=False))
print()
print(te_df[te_df.label == 1].groupby("fake_type")[["old_caught", "new_caught"]].mean().round(3).to_markdown())

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

| model                                |   acc |   prec |   recall |    f1 |   auc |
|:-------------------------------------|------:|-------:|---------:|------:|------:|
| Features (old, 9 features)           | 0.919 |  0.969 |    0.866 | 0.914 | 0.985 |
| Features + skills (new, 12 features) | 0.988 |  0.994 |    0.982 | 0.988 | 0.999 |

| fake_type          |   old_caught |   new_caught |
|:-------------------|-------------:|-------------:|
| generic_padding    |        1     |        1     |
| keyword_stuffing   |        1     |        1     |
| unsupported_skills |        0.599 |        0.946 |


In [18]:
# ===== DAY 2A: unseen fake types + length-bias check =====
import random
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

need = ["data", "feat_df", "feat_df2", "tr", "te", "y", "gb_old", "gb_new", "score",
        "extract_features", "more_features", "skill_features", "insert_at", "SKILLS"]
missing_names = [n for n in need if n not in globals()]
if missing_names:
    raise RuntimeError("Run the Day 1 cell first (it creates: " + ", ".join(missing_names) + ")")

random.seed(7)
OLD_COLS = list(feat_df.columns)
ALL_COLS = list(feat_df2.columns)

def featurize(texts):
    df = pd.DataFrame([{**extract_features(t), **more_features(t), **skill_features(t)} for t in texts])
    return df[ALL_COLS]

# ---- Three NEW fake types that the model has never seen in training ----
def add_to_experience(t, line):
    lines = t.split("\n")
    if "EXPERIENCE" in lines:
        i = lines.index("EXPERIENCE") + 1
        lines.insert(i, line)
        return "\n".join(lines)
    return insert_at(t, line, 0.7)

def spread_skills(t, k=8):
    """Claims extra skills in the SKILLS list AND mentions them once in EXPERIENCE (so they look 'supported')."""
    missing = [s for s in SKILLS if s.lower() not in t.lower()]
    extra = random.sample(missing, min(k, len(missing)))
    if "SKILLS" in t.split("\n"):
        t = t.replace("SKILLS\n", "SKILLS\n" + ", ".join(extra) + ", ", 1)
    else:
        t = insert_at(t, "Skills: " + ", ".join(extra) + ".", 0.3)
    return add_to_experience(t, "Worked with " + ", ".join(extra) + " in several projects.")

def duplicate_section(t):
    """Copy-pastes the EXPERIENCE block twice."""
    lines = t.split("\n")
    if "EXPERIENCE" in lines:
        i = lines.index("EXPERIENCE") + 1
        j = i
        while j < len(lines) and not (lines[j].isupper() and len(lines[j].split()) <= 3):
            j += 1
        block = lines[i:j]
        if block:
            return "\n".join(lines + block)
    return "\n".join(lines + lines[len(lines) // 2:])

TITLES = ["Chief Technology Officer", "Director of Engineering", "Senior Principal Architect",
          "VP of Data Science", "Head of AI"]
def title_inflation(t):
    """Adds invented senior job titles."""
    line = "Served as " + random.choice(TITLES) + " and " + random.choice(TITLES) + " leading global teams."
    return add_to_experience(t, line)

UNSEEN = {"spread_skills": spread_skills, "duplicate_section": duplicate_section, "title_inflation": title_inflation}

genuine_te = data.iloc[te][data.iloc[te].label == 0].text.tolist()
print("genuine test resumes used:", len(genuine_te))

rows = []
for name, fn in UNSEEN.items():
    X = featurize([fn(t) for t in genuine_te])
    rows.append({"unseen fake type": name,
                 "caught by old model (9 features)": float((gb_old.predict_proba(X[OLD_COLS])[:, 1] > .5).mean()),
                 "caught by new model (12 features)": float((gb_new.predict_proba(X)[:, 1] > .5).mean())})
print("\n=== 1) Share of UNSEEN fake types caught ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---- Length-bias check ----
cols_nolen = [c for c in ALL_COLS if c != "n_words"]
gb_nolen = GradientBoostingClassifier(random_state=42).fit(feat_df2.iloc[tr][cols_nolen], y[tr])

rng = np.random.RandomState(0)
all_genuine = data[data.label == 0].text.values
long_genuine = [t + "\n" + all_genuine[rng.randint(len(all_genuine))] for t in genuine_te]   # roughly 2x longer
X_norm = featurize(genuine_te)
X_long = featurize(long_genuine)

def false_alarm(model, X, cols):
    return float((model.predict_proba(X[cols])[:, 1] > .5).mean())

print("\n=== 2) Length-bias check: genuine resumes wrongly flagged ===")
print(pd.DataFrame([
    {"model": "with n_words (current)", "normal length": false_alarm(gb_new, X_norm, ALL_COLS), "2x longer": false_alarm(gb_new, X_long, ALL_COLS)},
    {"model": "without n_words",        "normal length": false_alarm(gb_nolen, X_norm, cols_nolen), "2x longer": false_alarm(gb_nolen, X_long, cols_nolen)},
]).round(3).to_string(index=False))

print("\n=== 3) Overall test results ===")
p_nolen = gb_nolen.predict_proba(feat_df2.iloc[te][cols_nolen])[:, 1]
print(pd.DataFrame([score("Features + skills (with n_words)", y[te], p_new),
                    score("Features + skills (without n_words)", y[te], p_nolen)]).round(3).to_string(index=False))

genuine test resumes used: 500

=== 1) Share of UNSEEN fake types caught ===
 unseen fake type  caught by old model (9 features)  caught by new model (12 features)
    spread_skills                             0.994                              0.338
duplicate_section                             0.392                              0.332
  title_inflation                             0.022                              0.010

=== 2) Length-bias check: genuine resumes wrongly flagged ===
                 model  normal length  2x longer
with n_words (current)          0.006      0.168
       without n_words          0.008      0.152

=== 3) Overall test results ===
                              model   acc  prec  recall    f1   auc
   Features + skills (with n_words) 0.988 0.994   0.982 0.988 0.999
Features + skills (without n_words) 0.988 0.992   0.984 0.988 1.000


In [19]:
# ===== DAY 2B: robust training + "leave one fake type out" test =====
import re, random
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier

need = ["data", "tr", "te", "y", "score", "UNSEEN", "featurize", "genuine_te", "long_genuine", "ALL_COLS"]
missing_names = [n for n in need if n not in globals()]
if missing_names:
    raise RuntimeError("Run the Day 1 cell and then the Day 2A cell first (missing: " + ", ".join(missing_names) + ")")

# ---- 1) Two extra features: copy-pasted lines, and senior-title words ----
SENIOR = {"chief", "director", "vp", "vice", "president", "head", "principal", "officer", "cto", "ceo"}
def robust_extra(t):
    lines = [l.strip().lower() for l in t.split("\n") if l.strip()]
    seen, dup = set(), 0
    for l in lines:
        if l in seen:
            dup += 1
        seen.add(l)
    toks = re.findall(r"[a-z]+", t.lower())
    n = max(len(toks), 1)
    return {"dup_line_ratio": dup / max(len(lines), 1),
            "senior_terms_per_100w": 100 * sum(w in SENIOR for w in toks) / n}

FEATS3 = [c for c in ALL_COLS if c != "n_words"] + ["dup_line_ratio", "senior_terms_per_100w"]   # n_words removed

def featurize3(texts):
    base_f = featurize(texts)
    extra = pd.DataFrame([robust_extra(t) for t in texts])
    return pd.concat([base_f.reset_index(drop=True), extra], axis=1)[FEATS3]

# ---- 2) Training data: original rows + extra fakes + long genuine resumes (all from TRAIN resumes only) ----
X_base = featurize3(data.iloc[tr].text.tolist())
y_base = y[tr]
gen_tr = data.iloc[tr][data.iloc[tr].label == 0].text.tolist()
rng2 = np.random.RandomState(5)
sub = [gen_tr[i] for i in rng2.choice(len(gen_tr), size=min(600, len(gen_tr)), replace=False)]

random.seed(11)
aug_X = {T: featurize3([fn(t) for t in sub]) for T, fn in UNSEEN.items()}
long_tr = [t + "\n" + gen_tr[rng2.randint(len(gen_tr))] for t in sub]       # genuine, about 2x longer
X_long_tr = featurize3(long_tr)

def make_train(types):
    Xs = [X_base] + [aug_X[T] for T in types] + [X_long_tr]
    ys = [y_base] + [np.ones(len(sub))] * len(types) + [np.zeros(len(sub))]
    return pd.concat(Xs, ignore_index=True), np.concatenate(ys)

# ---- 3) Test data (different resumes, different random draws) ----
random.seed(99)
test_fake_X = {T: featurize3([fn(t) for t in genuine_te]) for T, fn in UNSEEN.items()}
X_norm3 = featurize3(genuine_te)
X_long3 = featurize3(long_genuine)
X_te3 = featurize3(data.iloc[te].text.tolist())

def rate(model, X):
    return float((model.predict_proba(X)[:, 1] > .5).mean())

# ---- 4) Leave-one-fake-type-out: how well does it catch a type it NEVER saw in training? ----
rows = []
for held in UNSEEN:
    types = [T for T in UNSEEN if T != held]
    Xtr, ytr = make_train(types)
    m = GradientBoostingClassifier(random_state=42).fit(Xtr, ytr)
    rows.append({"fake type held out": held,
                 "caught (never seen)": rate(m, test_fake_X[held]),
                 "false alarm, normal": rate(m, X_norm3),
                 "false alarm, 2x longer": rate(m, X_long3)})
print("=== 1) Leave-one-fake-type-out (honest test on unseen fake types) ===")
print(pd.DataFrame(rows).round(3).to_string(index=False))

# ---- 5) Final model: trained on everything ----
Xtr, ytr = make_train(list(UNSEEN))
final_model = GradientBoostingClassifier(random_state=42).fit(Xtr, ytr)

print("\n=== 2) Final model: share of fakes caught (these types WERE in training) ===")
te_types = data.iloc[te].fake_type.values
p_te = final_model.predict_proba(X_te3)[:, 1]
out = []
for ft in ["keyword_stuffing", "unsupported_skills", "generic_padding"]:
    out.append({"fake type": ft, "caught": float((p_te[(te_types == ft)] > .5).mean())})
for T in UNSEEN:
    out.append({"fake type": T, "caught": rate(final_model, test_fake_X[T])})
print(pd.DataFrame(out).round(3).to_string(index=False))

print("\n=== 3) Final model: genuine resumes wrongly flagged ===")
print(pd.DataFrame([{"normal length": rate(final_model, X_norm3), "2x longer": rate(final_model, X_long3)}]).round(3).to_string(index=False))

print("\n=== 4) Final model: overall results on the ORIGINAL test set ===")
print(pd.DataFrame([score("Final model (14 features)", y[te], p_te)]).round(3).to_string(index=False))

=== 1) Leave-one-fake-type-out (honest test on unseen fake types) ===
fake type held out  caught (never seen)  false alarm, normal  false alarm, 2x longer
     spread_skills                0.484                0.056                   0.022
 duplicate_section                0.000                0.044                   0.000
   title_inflation                0.010                0.010                   0.028

=== 2) Final model: share of fakes caught (these types WERE in training) ===
         fake type  caught
  keyword_stuffing   1.000
unsupported_skills   0.958
   generic_padding   1.000
     spread_skills   0.990
 duplicate_section   0.992
   title_inflation   0.926

=== 3) Final model: genuine resumes wrongly flagged ===
 normal length  2x longer
         0.046      0.026

=== 4) Final model: overall results on the ORIGINAL test set ===
                    model  acc  prec  recall   f1   auc
Final model (14 features) 0.97 0.955   0.986 0.97 0.996


In [20]:
# ===== DAY 2C: save the final model + check REAL resumes =====
import io, sys, subprocess, zipfile, joblib
import pandas as pd
from google.colab import files
try:
    import pdfplumber
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pdfplumber"])
    import pdfplumber

need = ["final_model", "featurize3", "FEATS3"]
missing_names = [n for n in need if n not in globals()]
if missing_names:
    raise RuntimeError("Run the Day 2B cell first (missing: " + ", ".join(missing_names) + ")")

# ---- 1) Save the final model (v4) and download it ----
joblib.dump(final_model, "authenticity_model_v4.pkl")
joblib.dump(FEATS3, "features_v4.pkl")
with zipfile.ZipFile("models_v4.zip", "w") as z:
    z.write("authenticity_model_v4.pkl")
    z.write("features_v4.pkl")
files.download("models_v4.zip")

# ---- 2) Score real resumes (prints only #1, #2... never names or text) ----
print("Upload 1 or more REAL resume PDFs (yours, and classmates' with their permission):")
up = files.upload()

rows = []
for i, (fn, content) in enumerate(up.items(), 1):
    text_i = ""
    with pdfplumber.open(io.BytesIO(content)) as pdf:
        for page in pdf.pages:
            t = page.extract_text()
            if t:
                text_i += t + "\n"
    if len(text_i.split()) < 30:
        rows.append({"resume": f"#{i}", "words": len(text_i.split()), "P(fake)": None,
                     "verdict": "too little text (scanned PDF?)"})
        continue
    p = float(final_model.predict_proba(featurize3([text_i]))[0, 1])
    rows.append({"resume": f"#{i}", "words": len(text_i.split()), "P(fake)": round(p, 3),
                 "verdict": "flagged" if p > .5 else "genuine"})

res = pd.DataFrame(rows)
print(res.to_string(index=False))
scored = res.dropna(subset=["P(fake)"])
print(f"\nflagged as fake: {(scored['P(fake)'] > .5).sum()} of {len(scored)}")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Upload 1 or more REAL resume PDFs (yours, and classmates' with their permission):


Saving Yash_Raj_Resume (1).pdf to Yash_Raj_Resume (1) (1).pdf
Saving Yash_Raj_Resume data anal.pdf to Yash_Raj_Resume data anal (1).pdf
resume  words  P(fake) verdict
    #1    351    0.146 genuine
    #2    362    0.050 genuine

flagged as fake: 0 of 2
